In [1]:
from copy import deepcopy
import random
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\zhiho\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


In [2]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



Device: cuda


In [3]:
import sys

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: c:\Users\zhiho\Desktop\AI_stuff\05_Projects\School\Computer Vision and Deep Learning - Term Project\04_Outputs\notebooks\.venv\Scripts\python.exe
PyTorch: 2.14.0+cu126
PyTorch CUDA build: 12.6
CUDA available: True
GPU: NVIDIA GeForce MX250


In [4]:
# Keep the paths and CSV loading procedure aligned with data_loading.ipynb.
DATA_ROOT = Path(
    r"C:\Users\zhiho\Desktop\AI_stuff\05_Projects\School"
    r"\Computer Vision and Deep Learning - Term Project"
    r"\05_Data"
)
MANIFEST_DIR = DATA_ROOT / "manifests"

TRAIN_CSV = MANIFEST_DIR / "train_classifier.csv"
VAL_CSV = MANIFEST_DIR / "val_classifier.csv"
TEST_CSV = MANIFEST_DIR / "test_classifier.csv"

dtype_map = {
    "source_id": str,
    "generator_source_id": str,
}

train_df = pd.read_csv(TRAIN_CSV, dtype=dtype_map)
val_df = pd.read_csv(VAL_CSV, dtype=dtype_map)
test_df = pd.read_csv(TEST_CSV, dtype=dtype_map)


In [5]:
class LSBClassificationDataset(Dataset):
    def __init__(self, dataframe, data_root, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.data_root = Path(data_root)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = self.data_root / row["image_path"]

        with Image.open(image_path) as image_file:
            image = image_file.convert("RGB")

        label = int(row["label"])
        if self.transform is not None:
            image = self.transform(image)

        return image, label


In [6]:
# Apply one orientation-only augmentation to each training image.
orientation_augmentations = transforms.RandomChoice([
    transforms.RandomRotation(degrees=(90, 90)),
    transforms.RandomRotation(degrees=(180, 180)),
    transforms.RandomRotation(degrees=(270, 270)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomVerticalFlip(p=1.0),
])

train_transform = transforms.Compose([
    orientation_augmentations,
    transforms.ToTensor(),
])
evaluation_transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = LSBClassificationDataset(
    dataframe=train_df,
    data_root=DATA_ROOT,
    transform=train_transform,
)
val_dataset = LSBClassificationDataset(
    dataframe=val_df,
    data_root=DATA_ROOT,
    transform=evaluation_transform,
)
test_dataset = LSBClassificationDataset(
    dataframe=test_df,
    data_root=DATA_ROOT,
    transform=evaluation_transform,
)


In [7]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)


## Four-layer CNN

The four learnable layers are `Conv2d(3, 8)`, `Conv2d(8, 16)`, `Conv2d(16, 32)`, and `Linear(32, 2)`. ReLU and max-pooling follow each convolution. Adaptive average pooling makes the output layer independent of the input image dimensions.

In [20]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out = out + self.shortcut(x)
        out = self.relu(out)
        return out

In [ ]:
class CNNBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.residual_block1 = ResidualBlock(3, 16, stride=1)
        self.residual_block2 = ResidualBlock(16, 16, stride=1)
        self.residual_block3 = ResidualBlock(16, 32, stride=1)
        
        self.residual_block4 = ResidualBlock(64, 128, stride=2)
        self.fc1 = nn.Linear(128 * 64 * 64, 64)
        self.fc2 = nn.Linear(64, 2)
        self.relu = nn.ReLU()

    def forward(self, images):
        images = self.residual_block1(images)
        images = self.residual_block2(images)
        images = self.residual_block3(images)
        images = self.residual_block4(images)
        
        images = images.view(images.size(0), -1)
        images = self.fc1(images)
        images = self.relu(images)
        images = self.fc2(images)
        return images


In [17]:
model = CNNBaseline().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

NUM_EPOCHS = 3
history = {
    "train_accuracy": [],
    "val_accuracy": [],
}


In [18]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        #loss is cross entropy loss
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            predictions = outputs.argmax(dim=1)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (predictions == labels).sum().item()
            total_samples += batch_size
            all_labels.append(labels.cpu())
            all_predictions.append(predictions.cpu())

    return (
        total_loss / total_samples,
        total_correct / total_samples,
        torch.cat(all_labels),
        torch.cat(all_predictions),
    )


In [19]:
best_val_accuracy = -1.0
best_model_weights = deepcopy(model.state_dict())

for epoch in range(NUM_EPOCHS):
    train_loss, train_accuracy = train_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_accuracy, _, _ = evaluate(
        model, val_loader, criterion, device
    )

    history["train_accuracy"].append(train_accuracy)
    history["val_accuracy"].append(val_accuracy)

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_weights = deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"train loss: {train_loss:.4f} | train accuracy: {train_accuracy:.4f} | "
        f"val loss: {val_loss:.4f} | val accuracy: {val_accuracy:.4f}"
    )

model.load_state_dict(best_model_weights)
print(f"Best validation accuracy: {best_val_accuracy:.4f}")


KeyboardInterrupt: 

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
plt.figure(figsize=(7, 4))
plt.plot(epochs, history["train_accuracy"], marker="o", label="Train")
plt.plot(epochs, history["val_accuracy"], marker="o", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and validation accuracy")
plt.xticks(list(epochs))
plt.ylim(0.0, 1.0)
plt.grid(True)
plt.legend()
plt.show()


## Final test evaluation

The best validation-accuracy checkpoint is evaluated once on the held-out test split. The confusion matrix uses rows for actual classes and columns for predictions, ordered as clean then stego. Precision, recall, and F1 treat stego as the positive class.

In [ ]:
test_loss, test_accuracy, test_labels, test_predictions = evaluate(
    model, test_loader, criterion, device
)

confusion_matrix = torch.bincount(
    2 * test_labels + test_predictions,
    minlength=4,
).reshape(2, 2)

tn = confusion_matrix[0, 0].item()
fp = confusion_matrix[0, 1].item()
fn = confusion_matrix[1, 0].item()
tp = confusion_matrix[1, 1].item()

precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1_score = (
    2 * precision * recall / (precision + recall)
    if precision + recall
    else 0.0
)

print(f"Test loss:      {test_loss:.4f}")
print(f"Test accuracy:  {test_accuracy:.4f}")
print(f"Stego precision: {precision:.4f}")
print(f"Stego recall:    {recall:.4f}")
print(f"Stego F1-score:  {f1_score:.4f}")
print("Confusion matrix (rows=actual, columns=predicted; clean, stego):")
print(confusion_matrix)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(confusion_matrix.numpy(), cmap="Blues")
ax.set_xticks([0, 1])
ax.set_xticklabels(["clean", "stego"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["clean", "stego"])
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
ax.set_title("Test confusion matrix")

for row in range(2):
    for column in range(2):
        ax.text(
            column,
            row,
            str(confusion_matrix[row, column].item()),
            ha="center",
            va="center",
            color="black",
        )

plt.tight_layout()
plt.show()
